## Sentimen Analysis usin Deep Learning model

In [20]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [22]:
df = pd.read_csv("C:\\Users\\92324\\Desktop\\assesment\\sentiment_analysis_end_to_end\\src\\data\\cleaned_sentiment_dataset.csv")

df = df[['review', 'sentiment']]  # keep only needed columns
df.dropna(inplace=True)

In [23]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['review'] = df['review'].apply(clean_text)

In [24]:
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

# optional: save mapping
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [25]:
X = df['review'].values
y = df['sentiment'].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train = tokenizer.texts_to_sequences(X_train_text)
X_test = tokenizer.texts_to_sequences(X_test_text)

In [27]:
max_len = 200

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [28]:
print(X_train.shape)
print(X_train.dtype)
print(y_train.dtype)

(640, 200)
int32
int64


In [30]:
import numpy as np

y_train = np.array(y_train).astype("int32")
y_test = np.array(y_test).astype("int32")

In [32]:
model = Sequential()

model.add(Embedding(
    input_dim=5000,
    output_dim=128,
    input_shape=(200,)   # 🔥 FIX
))

model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

model.add(Dense(3, activation='softmax'))  # 3 classes!

In [33]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [34]:
model.build(input_shape=(None, 200))
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,971 (2.94 MB)

 Trainable params: 771,971 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/50


10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 797ms/step - accuracy: 0.8125 - loss: 0.3733 - val_accuracy: 0.7625 - val_loss: 0.4359
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 936ms/step - accuracy: 0.9125 - loss: 0.3053 - val_accuracy: 0.7312 - val_loss: 0.4326
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 945ms/step - accuracy: 0.9328 - loss: 0.2183 - val_accuracy: 0.7625 - val_loss: 0.4311
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.9609 - loss: 0.1359 - val_accuracy: 0.7875 - val_loss: 0.4546
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 980ms/step - accuracy: 0.9781 - loss: 0.0765 - val_accuracy: 0.7937 - val_loss: 0.4613
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 851ms/step - accuracy: 0.9906 - loss: 0.0484 - val_accuracy: 0.7812 - val_loss: 0.4937
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 667ms/step - accuracy: 0.9953 - loss: 0.0214 - val_accuracy: 0.8250 - val_loss: 0.5342
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 656ms/step - accuracy: 1.0000 - loss: 0.0113 - val_accuracy: 0.7688 - val_

In [42]:
model.save("sentiment_lstm.h5")

In [43]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [44]:
def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=200)
    
    pred = model.predict(padded)[0][0]
    
    return "positive" if pred > 0.5 else "negative"

In [45]:
def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=200)

    pred = model.predict(padded)[0]

    label_index = np.argmax(pred)
    confidence = np.max(pred)

    label = le.inverse_transform([label_index])[0]

    return f"{label} (confidence: {confidence:.2f})"

In [46]:
print(predict_sentiment("The service was amazing"))
print(predict_sentiment("I am very disappointed with this product"))
print(predict_sentiment("It is okay, nothing special"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
neutral (confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
neutral (confidence: 1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
neutral (confidence: 1.00)
